# CNV–PFI Association Analysis

InferCNV copy-number alterations and progression-free interval (PFI) category in HGSC

Author: Franziska Niemeyer

In [ ]:
ADATA_PATH = "../../../quality_control/primary-cohort/adata.h5ad"

CNV_FILES = {
    "BST2 amp"  : "../outputs/BST2_results/BST2_amplification_spots_annotated.tsv"}

PFI_CAT_COL = "PFI"
SAMPLE_COL  = "patient"

PFI_ORDER   = ["short", "medium", "long"]
PFI_PALETTE = {'short': '#C7844A', 'medium': '#456EAE', 'long': '#538984'}

MIN_SPOTS_PER_SAMPLE = 10   # samples with fewer spots are excluded
ALPHA                = 0.05

OUT_DIR = "../outputs/associations"
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from itertools   import combinations
from scipy.stats import kruskal, mannwhitneyu, spearmanr, chi2_contingency

from statsmodels.stats.multitest        import multipletests
from statsmodels.miscmodels.ordinal_model import OrderedModel

import anndata as ad

sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "legend.fontsize":    7,
    "legend.title_fontsize": 8,
    "figure.titlesize":   13,
    "figure.titleweight": "bold",
    "figure.dpi":         300,
})

### Load PFI category and spot counts from AnnData

One row per sample: PFI category and total spot count.

In [ ]:
def load_metadata_from_adata(path, pfi_cat_col, sample_col):
    """Return a per-sample DataFrame with PFI category and spot count."""
    adata = ad.read_h5ad(path)
    adata = adata[~adata.obs['patient'].isin(["Marker"])].copy()
    obs   = adata.obs[[pfi_cat_col]].copy()

    if sample_col and sample_col in adata.obs.columns:
        obs["patient"] = adata.obs[sample_col].astype(str)
    else:
        obs["patient"] = adata.obs.index.str.extract(r"-(\d+)$")[0]
        if obs["patient"].isna().any():
            raise ValueError(
                "Could not extract patient suffix from barcodes. "
                "Please set SAMPLE_COL to the correct obs column."
            )

    meta = (
        obs.groupby("patient")
           .agg(
               PFI_cat = (pfi_cat_col, "first"),
               n_spots = ("patient",    "count"),
           )
           .reset_index()
    )
    return meta, adata


import os

meta_df, adata = load_metadata_from_adata(ADATA_PATH, PFI_CAT_COL, SAMPLE_COL)
meta_df = meta_df[~meta_df["patient"].isin(['Marker'])].copy()

# Validate categories
meta_df["PFI_cat"] = meta_df["PFI_cat"].str.lower().str.strip()
unexpected = set(meta_df["PFI_cat"].unique()) - set(PFI_ORDER)
if unexpected:
    print(f"WARNING: unexpected PFI categories: {unexpected}")
    print(f"Expected one of: {PFI_ORDER}")

# Make an ordered categorical
meta_df["PFI_cat"] = pd.Categorical(
    meta_df["PFI_cat"], categories=PFI_ORDER, ordered=True
)
# Numeric encoding for trend tests (0 = short, 1 = medium, 2 = long)
meta_df["PFI_num"] = meta_df["PFI_cat"].cat.codes

print()
print(meta_df.to_string(index=False))
print()
print("PFI category counts:")
print(meta_df["PFI_cat"].value_counts()[PFI_ORDER])

### Compute per-sample CNV fractions

For each gene/CNV:
- **n_cnv_spots**: spots carrying the CNV in that sample
- **cnv_fraction**: n_cnv_spots / total spots in sample


In [ ]:
def build_cnv_fraction_table(cnv_file, meta_df, min_spots=10, cnv_label="CNV"):
    """
    Load a *_spots_annotated.tsv, compute per-sample CNV fraction,
    merge with metadata, and return a tidy DataFrame.
    """
    spots = pd.read_csv(cnv_file, sep="\t")
    spots["patient"] = spots["subcluster"].str.extract(r"^(H\d+)")

    cnv_counts = (
        spots.groupby("patient")
             .size()
             .reset_index(name="n_cnv_spots")
    )

    df = meta_df.merge(cnv_counts, on="patient", how="left")
    df["n_cnv_spots"] = df["n_cnv_spots"].fillna(0).astype(int)
    df = df[df["n_spots"] >= min_spots].copy()
    df["cnv_fraction"] = df["n_cnv_spots"] / df["n_spots"]
    df["cnv_label"]    = cnv_label
    return df


cnv_tables = {}
for label, path in CNV_FILES.items():
    if not os.path.exists(path):
        print(f"  WARNING: {path} not found — skipping {label}")
        continue
    cnv_tables[label] = build_cnv_fraction_table(
        path, meta_df, min_spots=MIN_SPOTS_PER_SAMPLE, cnv_label=label
    )
    df = cnv_tables[label]
    print(f"  {label:<12}  n={len(df)}")
    for cat in PFI_ORDER:
        sub = df[df["PFI_cat"] == cat]
        if len(sub):
            print(f"    {cat:<8} n={len(sub)}  "
                  f"median CNV frac={sub['cnv_fraction'].median():.2f}  "
                  f"range=[{sub['cnv_fraction'].min():.2f}–{sub['cnv_fraction'].max():.2f}]")

n_cnv = len(cnv_tables)

### CNV fraction overview

In [ ]:
if n_cnv == 0:
    print("No CNV files loaded — check CNV_FILES paths.")
else:
    fig, axes = plt.subplots(1, n_cnv, figsize=(4.5, 3), sharey=False)
    if n_cnv == 1:
        axes = [axes]

    for ax, (label, df) in zip(axes, cnv_tables.items()):
        df_sorted = df.sort_values("cnv_fraction", ascending=False)
        colors = [PFI_PALETTE[str(g)] for g in df_sorted["PFI_cat"]]
        ax.bar(df_sorted["patient"], df_sorted["cnv_fraction"],
               color=colors, edgecolor="white", linewidth=0.5, zorder=2)
        ax.set_title("Per-sample chr19p13.2 gain burden")
        ax.set_xlabel("Patient")
        ax.set_ylabel("Fraction of spots with CNV")
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=45)

    # Shared legend
    legend_patches = [
        mpatches.Patch(facecolor=PFI_PALETTE[c], label=f"PFI: {c}")
        for c in PFI_ORDER
    ]
    fig.legend(handles=legend_patches, loc="center left",
               bbox_to_anchor=(1., .5), title="PFI category")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/cnv_fraction_overview.pdf", bbox_inches="tight")
    plt.savefig(f"{OUT_DIR}/cnv_fraction_overview.png", dpi=600, bbox_inches="tight")
    plt.show()

### CNV fraction per PFI category

Visualises the distribution of CNV fraction within each PFI category.
Individual samples are labelled to account for the small cohort size.


In [ ]:
if n_cnv == 0:
    print("No CNV data loaded.")
else:
    fig, axes = plt.subplots(1, n_cnv, figsize=(3, 3), sharey=False)
    if n_cnv == 1:
        axes = [axes]

    for ax, (label, df) in zip(axes, cnv_tables.items()):
        sns.boxplot(data=df, x="PFI_cat", y="cnv_fraction",
                    order=PFI_ORDER, palette=PFI_PALETTE,
                    width=0.5, ax=ax, boxprops=dict(alpha=0.65),
                    fliersize=0)
        sns.stripplot(data=df, x="PFI_cat", y="cnv_fraction",
                      order=PFI_ORDER, palette=PFI_PALETTE,
                      size=5, jitter=True, ax=ax,
                      edgecolor="white", linewidth=0.5, zorder=3)

        for _, row in df.iterrows():
            xpos = PFI_ORDER.index(str(row["PFI_cat"]))
            ax.annotate(
                row["patient"],
                xy=(xpos, row["cnv_fraction"]),
                xytext=(6, 0), textcoords="offset points",
                fontsize=7, color="#444444", va="center"
            )

        ax.set_title("Chr19 gain involving BST2", fontsize=12, fontweight="bold")
        ax.set_xlabel("PFI category")
        ax.set_ylabel("CNV fraction (spots)")
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/cnv_fraction_boxplots.pdf", bbox_inches="tight")
    plt.savefig(f"{OUT_DIR}/cnv_fraction_boxplots.png", dpi=600, bbox_inches="tight")
    plt.show()

## Kruskal–Wallis test

Non-parametric one-way ANOVA across the three PFI categories.
Tests the global null hypothesis that CNV fraction is identically distributed in short, medium, and long PFI groups.

In [ ]:
kw_results = {}

for label, df in cnv_tables.items():
    groups = [
        df.loc[df["PFI_cat"] == cat, "cnv_fraction"].values
        for cat in PFI_ORDER
        if (df["PFI_cat"] == cat).sum() >= 2
    ]
    valid_cats = [
        cat for cat in PFI_ORDER
        if (df["PFI_cat"] == cat).sum() >= 2
    ]

    if len(groups) < 2:
        print(f"  {label}: not enough groups with n>=2 for Kruskal-Wallis")
        continue

    stat, p = kruskal(*groups)
    eta_sq   = (stat - len(groups) + 1) / (len(df) - len(groups))  # eta-squared approx
    eta_sq   = max(0, eta_sq)

    kw_results[label] = {"stat": stat, "p": p, "eta_sq": eta_sq,
                          "valid_cats": valid_cats}
    sig = " *" if p < ALPHA else (" (trend)" if p < 0.10 else "")
    print(f"  {label:<12}  H={stat:.3f}  p={p:.4f}{sig}  "
          f"eta²={eta_sq:.3f}  groups={valid_cats}")

# BH correction across all CNVs
if kw_results:
    kw_df = pd.DataFrame([
        {"CNV": k, "H_stat": v["stat"], "p_kruskal": v["p"], "eta_sq": v["eta_sq"]}
        for k, v in kw_results.items()
    ])
    _, kw_df["p_adj_BH"], _, _ = multipletests(kw_df["p_kruskal"], method="fdr_bh")
    kw_df["significant_raw"] = kw_df["p_kruskal"] < ALPHA
    kw_df["significant_BH"]  = kw_df["p_adj_BH"]  < ALPHA
    print("\nKruskal-Wallis summary with BH correction:")
    print(kw_df.round(4).to_string(index=False))
    kw_df.to_csv(f"{OUT_DIR}/kruskal_wallis_results.csv", index=False)
    print(f"\nSaved: {OUT_DIR}/kruskal_wallis_results.csv")

In [ ]:
ordinal_results = []

for label, df in cnv_tables.items():
    fit_df = df[["PFI_cat", "cnv_fraction"]].dropna()
    if len(fit_df) < 5:
        print(f"  {label}: too few samples (n={len(fit_df)}) — skipping")
        continue

    try:
        mod = OrderedModel(
            fit_df["PFI_cat"],
            fit_df[["cnv_fraction"]],
            distr="logit"
        )
        res = mod.fit(method="bfgs", disp=False)

        coef = res.params["cnv_fraction"]
        se   = res.bse["cnv_fraction"]
        z    = res.tvalues["cnv_fraction"]
        p    = res.pvalues["cnv_fraction"]
        OR   = np.exp(coef)
        ci_lo = np.exp(coef - 1.96 * se)
        ci_hi = np.exp(coef + 1.96 * se)

        ordinal_results.append({
            "CNV"     : label,
            "coef"    : round(coef, 3),
            "OR"      : round(OR, 3),
            "OR_CI_lo": round(ci_lo, 3),
            "OR_CI_hi": round(ci_hi, 3),
            "z"       : round(z, 3),
            "p_value" : round(p, 4),
            "n"       : len(fit_df),
        })
        sig = " *" if p < ALPHA else (" (trend)" if p < 0.10 else "")
        print(f"  {label:<12}  OR={OR:.3f} [{ci_lo:.3f}–{ci_hi:.3f}]  "
              f"p={p:.4f}{sig}  n={len(fit_df)}")

    except Exception as e:
        print(f"  {label}: ordinal model failed — {e}")

if ordinal_results:
    ord_df = pd.DataFrame(ordinal_results)
    _, ord_df["p_adj_BH"], _, _ = multipletests(ord_df["p_value"], method="fdr_bh")
    print("\nOrdinal logistic regression summary with BH correction:")
    print(ord_df.round(4).to_string(index=False))
    ord_df.to_csv(f"{OUT_DIR}/ordinal_regression_results.csv", index=False)
    print(f"\nSaved: {OUT_DIR}/ordinal_regression_results.csv")


## CNV co-occurrence

Samples (rows) × CNVs (columns), coloured by CNV fraction.
Row side-colours indicate PFI category. Hierarchical clustering reveals samples with similar CNV burden profiles.


In [ ]:
if len(cnv_tables) >= 2:
    frac_wide = pd.DataFrame(index=meta_df["sample"])
    for label, df in cnv_tables.items():
        frac_wide[label] = df.set_index("sample")["cnv_fraction"]
    frac_wide = frac_wide.fillna(0)

    row_pfi    = meta_df.set_index("sample")["PFI_cat"].reindex(frac_wide.index)
    row_colors = row_pfi.map(PFI_PALETTE)
    row_colors.name = "PFI category"

    g = sns.clustermap(
        frac_wide,
        cmap="RdBu_r", center=0.5, vmin=0, vmax=1,
        row_colors=row_colors,
        annot=True, fmt=".2f", annot_kws={"size": 9},
        figsize=(max(6, len(cnv_tables) * 2), max(5, len(frac_wide) * 0.8)),
        linewidths=0.5,
        cbar_kws={"label": "Fraction of spots with CNV", "shrink": 0.5},
    )
    # Add PFI category legend
    legend_patches = [
        mpatches.Patch(facecolor=PFI_PALETTE[c], label=f"PFI: {c}")
        for c in PFI_ORDER
    ]
    g.fig.legend(handles=legend_patches, loc="upper left",
                 bbox_to_anchor=(0.01, 0.99), fontsize=9,
                 title="PFI category", frameon=True)
    g.fig.suptitle("CNV fraction heatmap — samples × CNVs", y=1.02,
                   fontsize=13, fontweight="bold")

    plt.savefig(f"{OUT_DIR}/cnv_cooccurrence_heatmap.pdf", bbox_inches="tight")
    plt.show()

## Spearman correlation: CNV fraction vs PFI category

Treats PFI category as numeric (short=0, medium=1, long=2) and computes Spearman's ρ against CNV fraction. A negative ρ means higher CNV fraction correlates with shorter PFI.

In [ ]:
if n_cnv == 0:
    print("No CNV data loaded.")
else:
    fig, axes = plt.subplots(1, n_cnv, figsize=(4.5, 4.5))
    if n_cnv == 1:
        axes = [axes]

    spearman_results = []
    for ax, (label, df) in zip(axes, cnv_tables.items()):
        x   = df["cnv_fraction"].values
        y   = df["PFI_num"].values      # 0=short, 1=medium, 2=long
        rho, p = spearmanr(x, y)
        spearman_results.append({"CNV": label, "rho": rho, "p": p})

        colors = [PFI_PALETTE[str(c)] for c in df["PFI_cat"]]
        ax.scatter(x, y, c=colors, s=80, edgecolors="white", linewidth=0.5, zorder=3)

        # Jitter y for visibility
        jitter = np.random.default_rng(42).uniform(-0.08, 0.08, len(y))
        ax.scatter(x, y + jitter, c=colors, s=60,
                   edgecolors="white", linewidth=0.5, zorder=3, alpha=0.0)

        if len(x) >= 3:
            m, b = np.polyfit(x, y, 1)
            xx = np.linspace(x.min(), x.max(), 100)
            ax.plot(xx, m*xx + b, color="gray", linewidth=1.5,
                    linestyle="--", alpha=0.7)

        for _, row in df.iterrows():
            ax.annotate(row["patient"],
                        (row["cnv_fraction"], row["PFI_num"]),
                        xytext=(5, 3), textcoords="offset points",
                        fontsize=7, color="#444444")

        p_str = f"ρ={rho:.2f}  p={p:.3f}" if p >= 0.001 else f"ρ={rho:.2f}  p<0.001"
        ax.set_title(f"{label}\nSpearman {p_str}", fontsize=10, fontweight="bold")
        ax.set_xlabel("CNV fraction")
        ax.set_ylabel("PFI category (numeric)")
        ax.set_yticks([0, 1, 2])
        ax.set_yticklabels(PFI_ORDER)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/05_spearman_scatter.pdf", bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR}/05_spearman_scatter.pdf")
